# QC Layer Ablation

This notebook measures the **incremental contribution of each QC tier** to recall,
F1, and zero-leak rate, and evaluates the **calibration of the model's intrinsic
`<qc_status>` tag**.

## What we measure

| Stage | What runs | What changes |
|-------|-----------|-------------|
| **Raw** | Model output only, no QC | Baseline |
| **+Repair** | XML structural repair only (no detectors) | Recover malformed XML |
| **+Tier-2** | Repair + regex detectors + reconcile | Recall floor on 6 tags |
| **+Tier-3** | Tier-2 + spaCy NER on escalated samples | Recall floor on name/place/address |

### QC Calibration
Separately, we audit how well `<qc_status>complete/incomplete</qc_status>` predicts
whether the model actually missed PHI.  The key number is the **FN rate** —
model says 'complete' but recall < 1.0 — which must be as low as possible.


In [ ]:
# --- repo bootstrap: makes this notebook runnable from anywhere ---
import os, sys
ROOT = os.getcwd()
while ROOT != os.path.dirname(ROOT) and not os.path.exists(os.path.join(ROOT, "pyproject.toml")):
    ROOT = os.path.dirname(ROOT)
if not os.path.exists(os.path.join(ROOT, "pyproject.toml")):
    ROOT = "/kaggle/input/datasets/sinadaneshgar/edgedeid"  # Kaggle dataset fallback
os.chdir(ROOT); sys.path.insert(0, ROOT)
print("repo root:", ROOT)

## 0. Setup

In [ ]:
import sys, os
# Adjust this path if your working directory differs
ROOT = os.getcwd()  # bootstrap already chdir'd to repo root
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

import pandas as pd
import numpy as np
import xml.etree.ElementTree as ET

from edgedeid.eval.metrics import compute_metrics, print_report, EvalResult
from edgedeid.qc import (
    QCConfig, apply_qc_batch,
    compute_qc_calibration, print_calibration_report, calibration_to_dict,
    repair_xml, empty_phi_xml,
)
from edgedeid.qc.reconcile import reconcile

print("Imports OK")

## 0.5 — Preflight smoke test  (run me first)\n\nValidates dataset, imports and the model-load path, and **downloads the spaCy Tier-3 model** before the run. Hard problems raise; optional gaps warn.

In [ ]:
# ── Preflight: fail fast + ensure spaCy for Tier-3 (run me first) ─────────────
import importlib, xml.etree.ElementTree as ET

_hard, _warn = [], []
def _ok(m):  print(f"  ok   {m}")
def _bad(m): _hard.append(m); print(f"  FAIL {m}")
def _wn(m):  _warn.append(m); print(f"  warn {m}")

print("[1] dataset + load path")
DATA = os.path.join(ROOT, "evals", "ext_eval", "revizied_evals.xlsx")
if not os.path.exists(DATA):
    _bad(f"missing dataset: {DATA}")
else:
    try:
        from edgedeid.data.preprocessing import preprocess_text
        from edgedeid.data.xml_utils import convert_to_xml
        _df = pd.read_excel(DATA)
        _miss = {"default_response_0", "default_response_1"} - set(_df.columns)
        if _miss:
            _bad(f"xlsx missing columns: {_miss}")
        else:
            _s = preprocess_text(_df.head(3).copy(),
                                 input_col="default_response_0", output_col="default_response_1")
            for _v in _s["default_response_1"].apply(convert_to_xml):
                ET.fromstring(_v)
            _ok(f"{len(_df)} rows · preprocess_text + convert_to_xml valid")
    except Exception as e:
        _bad(f"load path broken: {type(e).__name__}: {e}")

print("[2] QC + calibration + model-load imports")
try:
    from edgedeid.eval.metrics import compute_metrics, print_report
    from edgedeid.qc import (QCConfig, apply_qc_batch, compute_qc_calibration,
                             print_calibration_report, repair_xml, empty_phi_xml)
    from edgedeid.qc.reconcile import reconcile
    from edgedeid.rl.data import format_preference_prompt
    from scripts.load_model_from_hf import load_model
    _ok("compute_metrics · QC pipeline · calibration · load_model · format_preference_prompt")
except Exception as e:
    _bad(f"import: {type(e).__name__}: {e}")

print("[3] ensure spaCy en_core_web_sm (Tier-3 NER)")
def _ensure_spacy(model="en_core_web_sm"):
    """Install spaCy + the model on first run so QC Tier-3 is available."""
    import importlib, subprocess, sys
    try:
        import spacy
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "spacy"], check=True)
        importlib.invalidate_caches(); import spacy
    try:
        spacy.load(model)
    except OSError:
        print(f"  ... downloading spaCy {model} (one-time)")
        subprocess.run([sys.executable, "-m", "spacy", "download", model], check=True)
        importlib.invalidate_caches(); spacy.load(model)
    return True
try:
    _ensure_spacy(); _ok("spaCy en_core_web_sm ready (QC Tier-3 will run)")
except Exception as e:
    _wn(f"spaCy en_core_web_sm unavailable ({type(e).__name__}) → Tier-3 stage falls back to Tier-2")

print("[4] model access (GRPO predictions in §1)")
try:
    import torch; _ok(f"torch CUDA={torch.cuda.is_available()}")
except Exception as e: _bad(f"torch missing (required to generate GRPO predictions): {e}")
if os.environ.get("HF_TOKEN") or os.environ.get("HUGGING_FACE_HUB_TOKEN"):
    _ok("HF token present (private EdgeDeID/Qwen3-0.6B-GRPO)")
else:
    _wn("no HF_TOKEN → GRPO checkpoint download will 401 unless predictions are cached")

print("\n" + "=" * 60)
if _hard:
    print(f"PREFLIGHT FAILED — {len(_hard)} hard issue(s); fix before running:")
    for _m in _hard: print("   -", _m)
    raise RuntimeError("preflight hard failures: " + "; ".join(_hard))
print(f"PREFLIGHT PASSED  ({len(_warn)} warning(s))")

## 1. Load data + generate GRPO predictions (revizied_evals.xlsx)

Uses the **external test set** `evals/ext_eval/revizied_evals.xlsx` (100 manually
de-identified radiology reports), the same benchmark as `baselines.ipynb` and
`ablation.ipynb`.

The `<qc_status>` tag this notebook audits is emitted **only** by the GRPO model
prompted with `GRPO_SYSTEM_PROMPT`, so predictions are generated **in-process**
from the GRPO checkpoint (raw output kept intact — `QwenDeidBaseline` is not used
here because it trims the tag). Predictions are cached to `artifacts/qc/` after
the first run.

Produces `source_notes`, `ground_truths`, and `predictions`.

In [ ]:
# ── Load the external test set (revizied_evals.xlsx) → source_notes, ground_truths ──
from edgedeid.data.preprocessing import preprocess_text
from edgedeid.data.xml_utils import convert_to_xml

# xlsx cols: default_response_0 = raw note, default_response_1 = inline PHI-tagged
# gold — preprocess_text's native input/output_col names, so no renaming needed.
EVAL_XLSX = os.path.join(ROOT, "evals", "ext_eval", "revizied_evals.xlsx")
raw_df = pd.read_excel(EVAL_XLSX)
raw_df = preprocess_text(raw_df, input_col="default_response_0", output_col="default_response_1")
raw_df["GT-XML"] = raw_df["default_response_1"].apply(convert_to_xml)

SRC_COL = "default_response_0"
for tag in ["<anonymized_report>", "</anonymized_report>", "<anonymized_report/>",
            "<raw_report>", "</raw_report>", "<raw_report/>"]:
    raw_df[SRC_COL] = raw_df[SRC_COL].str.replace(tag, "", regex=False).str.strip()

source_notes  = raw_df[SRC_COL].tolist()
ground_truths = raw_df["GT-XML"].tolist()
print(f"Loaded {len(source_notes)} samples from revizied_evals.xlsx")

In [ ]:
# ── Generate raw GRPO predictions (with <qc_status>) in-process ───────────────
# Raw generation (not QwenDeidBaseline) so the <qc_status> tag that appears AFTER
# </phi_entities> is preserved. Cached after the first run.
import gc
import re as _re

import torch
from tqdm.auto import tqdm

QC_MODEL   = "grpo"
PRED_CACHE = os.path.join(ROOT, "artifacts", "qc", "en_predictions_grpo.csv")
os.makedirs(os.path.dirname(PRED_CACHE), exist_ok=True)

predictions = None
if os.path.exists(PRED_CACHE):
    cached = pd.read_csv(PRED_CACHE)
    if len(cached) == len(source_notes):
        predictions = cached["prediction"].fillna("").tolist()
        print(f"Loaded {len(predictions)} cached predictions from {PRED_CACHE}")
    else:
        print("Cache size mismatch — regenerating.")

if predictions is None:
    from scripts.load_model_from_hf import load_model
    from edgedeid.rl.data import format_preference_prompt

    model, tok = load_model(QC_MODEL)
    model.eval()
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token

    predictions = []
    for note in tqdm(source_notes, desc=f"{QC_MODEL} (raw, +qc_status)"):
        prompt = format_preference_prompt(note)
        enc = tok(prompt, return_tensors="pt", truncation=True, max_length=4096).to(model.device)
        with torch.no_grad():
            out = model.generate(**enc, max_new_tokens=1024, do_sample=False,
                                 pad_token_id=tok.pad_token_id)
        predictions.append(
            tok.decode(out[0][enc["input_ids"].shape[1]:], skip_special_tokens=True).strip()
        )

    pd.DataFrame({"note": source_notes, "prediction": predictions}).to_csv(PRED_CACHE, index=False)
    print(f"Generated + cached {len(predictions)} predictions -> {PRED_CACHE}")
    del tok; model = None; gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

n_qc = sum(bool(_re.search(r"<qc_status\s*>(complete|incomplete)</qc_status>", p, _re.I))
           for p in predictions)
print(f"  predictions carrying a <qc_status> tag: {n_qc}/{len(predictions)}")
print(f"  Predictions : {len(predictions)}")
print(f"  GT          : {len(ground_truths)}")
print(f"  Notes       : {len(source_notes)}")

## 2. Helper: produce per-stage prediction lists

In [ ]:
def repair_only(preds: list[str]) -> list[str]:
    """Tier-1 gate: structural XML repair, no detectors, no injection."""
    out = []
    for p in preds:
        r = repair_xml(p)
        out.append(r.xml if r.xml else empty_phi_xml())
    return out


def with_tier2(preds: list[str], notes: list[str]) -> tuple[list[str], list]:
    """Repair + Tier-2 regex detectors + reconcile (NO Tier-3)."""
    cfg = QCConfig(use_tier3=False)
    finals, results = apply_qc_batch(preds, notes, config=cfg)
    return finals, results


def with_tier3(preds: list[str], notes: list[str]) -> tuple[list[str], list]:
    """Repair + Tier-2 + Tier-3 spaCy NER on escalated samples."""
    cfg = QCConfig(use_tier3=True, tier3_on_escalated_only=True)
    finals, results = apply_qc_batch(preds, notes, config=cfg)
    return finals, results


print("Helper functions ready.")

## 3. Run each QC stage

In [ ]:
print("Running: Raw (no QC)...")
preds_raw = predictions

print("Running: +Repair only...")
preds_repair = repair_only(predictions)

print("Running: +Tier-2 (regex)...")
preds_t2, results_t2 = with_tier2(predictions, source_notes)

print("Running: +Tier-3 (spaCy NER)...")
try:
    preds_t3, results_t3 = with_tier3(predictions, source_notes)
    tier3_available = True
except ImportError as e:
    print(f"  [SKIP] Tier-3 unavailable: {e}")
    preds_t3, results_t3 = preds_t2, results_t2
    tier3_available = False

print("Done.")

## 4. Compute metrics per stage

In [ ]:
stages = [
    ("Raw (no QC)",      preds_raw),
    ("+Repair",          preds_repair),
    ("+Tier-2 (regex)",  preds_t2),
]
if tier3_available:
    stages.append(("+Tier-3 (spaCy)", preds_t3))

stage_results: dict[str, EvalResult] = {}
for label, preds in stages:
    r = compute_metrics(preds, ground_truths, source_notes=source_notes)
    stage_results[label] = r
    print_report(r, title=label)
    print()

## 5. Per-layer delta table

In [ ]:
rows = []
prev = None
for label, r in stage_results.items():
    row = {
        "Stage":         label,
        "Micro-P":       round(r.micro_precision, 4),
        "Micro-R":       round(r.micro_recall, 4),
        "Micro-F1":      round(r.micro_f1, 4),
        "F-beta(b=2)":   round(r.f_beta, 4),
        "Zero-leak":     round(r.zero_leak_rate, 4),
        "Halluc.": round(r.hallucination_rate, 4),
    }
    if prev is not None:
        row["ΔRecall"]    = round(r.micro_recall    - prev.micro_recall,    4)
        row["ΔZero-leak"] = round(r.zero_leak_rate  - prev.zero_leak_rate,  4)
        row["ΔF1"]        = round(r.micro_f1        - prev.micro_f1,        4)
    else:
        row["ΔRecall"] = row["ΔZero-leak"] = row["ΔF1"] = "—"
    rows.append(row)
    prev = r

delta_df = pd.DataFrame(rows)
print("=" * 100)
print(" QC Layer-by-Layer Delta")
print("=" * 100)
print(delta_df.to_string(index=False))
delta_df

## 6. Tier-2 escalation summary

In [ ]:
n = len(results_t2)
n_escalated   = sum(r.escalated for r in results_t2)
n_repaired    = sum(r.repair_status == "repaired" for r in results_t2)
n_unrecov     = sum(r.repair_status == "unrecoverable" for r in results_t2)
n_injected    = sum(r.n_injected for r in results_t2)
n_incomplete  = sum(r.intrinsic_qc == "incomplete" for r in results_t2)
n_no_tag      = sum(r.intrinsic_qc is None for r in results_t2)

print(f"Tier-2 Summary ({n} samples):")
print(f"  Escalated              : {n_escalated:5d}  ({100*n_escalated/n:.1f}%)")
print(f"  XML repaired           : {n_repaired:5d}")
print(f"  XML unrecoverable      : {n_unrecov:5d}")
print(f"  Model self-flagged     : {n_incomplete:5d}  (intrinsic=incomplete)")
print(f"  No <qc_status> tag     : {n_no_tag:5d}")
print(f"  Total T2 injections    : {n_injected:5d}  ({n_injected/n:.2f} per note avg)")

# Reason breakdown
from collections import Counter
reason_counts = Counter(reason for r in results_t2 for reason in r.reasons)
print("\n  Escalation reason counts:")
for reason, count in reason_counts.most_common():
    print(f"    {reason:<45} : {count}")

## 7. Tier-3 injection summary (escalated samples only)

In [ ]:
if tier3_available:
    n3_escalated = sum(r.escalated for r in results_t3)
    n3_injected  = sum(r.n_injected_tier3 for r in results_t3)
    by_type      = Counter(
        det_type
        for r in results_t3
        for det_type, _ in r.audit.get("injected_t3", [])
    )
    print(f"Tier-3 Summary ({len(results_t3)} samples):")
    print(f"  Samples that ran Tier-3  : {n3_escalated}")
    print(f"  Total T3 injections      : {n3_injected}")
    print(f"  Injections by type:")
    for t, c in by_type.most_common():
        print(f"    {t:<12} : {c}")
else:
    print("Tier-3 was not run (spaCy not installed).")

## 8. QC Calibration: how well does `<qc_status>` predict actual completeness?

We treat the `<qc_status>` tag as a binary classifier over each note:
- **Positive** = the model flagged its own output as `incomplete`
- **Actual label** = per-note recall < completeness_threshold

The **FN rate** (confident but missed PHI) is the safety-critical metric.

In [ ]:
COMPLETENESS_THRESHOLD = 0.99  # recall >= this → "actually complete"

calib = compute_qc_calibration(
    predictions, ground_truths,
    completeness_threshold=COMPLETENESS_THRESHOLD,
)
print_calibration_report(calib, title="QC CALIBRATION (raw model predictions)")

### 8.1 Threshold sensitivity: FN rate at different completeness thresholds

In [ ]:
thresholds = [0.80, 0.85, 0.90, 0.95, 0.99, 1.00]
rows_thresh = []
for th in thresholds:
    c = compute_qc_calibration(predictions, ground_truths, completeness_threshold=th)
    rows_thresh.append({
        "threshold": th,
        "n_actually_incomplete": c.n_actually_incomplete,
        "n_actually_complete":   c.n_actually_complete,
        "qc_recall":   round(c.qc_recall, 4),
        "qc_precision": round(c.qc_precision, 4),
        "fn_rate":  round(c.fn_rate, 4),
        "fp_rate":  round(c.fp_rate, 4),
        "escalation_rate": round(c.escalation_rate, 4),
    })

thresh_df = pd.DataFrame(rows_thresh)
print(thresh_df.to_string(index=False))
thresh_df

### 8.2 Completeness score distribution (histogram)

In [ ]:
try:
    import matplotlib.pyplot as plt
    scores = calib.scores
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    axes[0].hist(scores, bins=40, color='steelblue', edgecolor='white')
    axes[0].axvline(COMPLETENESS_THRESHOLD, color='red', linestyle='--',
                    label=f'threshold={COMPLETENESS_THRESHOLD}')
    axes[0].set_title('Completeness score distribution')
    axes[0].set_xlabel('reward_completeness (per note)')
    axes[0].set_ylabel('count')
    axes[0].legend()

    # Color by qc_status: green=correct, red=FN, orange=FP
    import re
    _QC_RE = re.compile(r'<qc_status\s*>(complete|incomplete)</qc_status>', re.I)
    colors = []
    for pred, score in zip(predictions, scores):
        m = _QC_RE.search(pred)
        claimed = m.group(1).lower() if m else 'incomplete'
        actual  = score >= COMPLETENESS_THRESHOLD
        if claimed == 'incomplete' and not actual:
            colors.append('green')    # TP
        elif claimed == 'complete' and actual:
            colors.append('blue')     # TN
        elif claimed == 'incomplete' and actual:
            colors.append('orange')   # FP
        else:
            colors.append('red')      # FN — dangerous

    axes[1].scatter(range(len(scores)), scores, c=colors, alpha=0.5, s=10)
    axes[1].axhline(COMPLETENESS_THRESHOLD, color='red', linestyle='--',
                    label=f'threshold={COMPLETENESS_THRESHOLD}')
    axes[1].set_title('Per-sample completeness (colored by QC outcome)')
    axes[1].set_xlabel('sample index')
    axes[1].set_ylabel('completeness score')
    from matplotlib.patches import Patch
    axes[1].legend(handles=[
        Patch(color='green',  label='TP (flagged, truly incomplete)'),
        Patch(color='blue',   label='TN (confident, truly complete)'),
        Patch(color='orange', label='FP (flagged, but complete)'),
        Patch(color='red',    label='FN (confident, missed PHI)'),
    ])
    plt.tight_layout()
    os.makedirs(os.path.join(ROOT, 'artifacts', 'qc'), exist_ok=True)
    plt.savefig(os.path.join(ROOT, 'artifacts', 'qc', 'qc_calibration.png'), dpi=150)
    plt.show()
    print("Figure saved to artifacts/qc/qc_calibration.png")
except ImportError:
    print("matplotlib not available — skipping plots.")

## 9. False Negative deep-dive

These are the **dangerous cases**: model said `complete` but actually missed PHI.

In [ ]:
import re as _re
from edgedeid.eval.metrics import extract_entities
from edgedeid.eval.metrics import _covers, _normalize

_QC_TAG = _re.compile(r'<qc_status\s*>(complete|incomplete)</qc_status>', _re.I)

fn_rows = []
for i, (pred, gt, note) in enumerate(zip(predictions, ground_truths, source_notes)):
    m = _QC_TAG.search(pred)
    if m and m.group(1).lower() == 'complete':
        # check per-sample completeness
        from edgedeid.rl.rewards import reward_completeness
        score = reward_completeness(pred, gt)
        if score < COMPLETENESS_THRESHOLD:
            # find which entities were missed
            from edgedeid.eval.metrics import extract_entities
            pred_ents = extract_entities(pred)
            gt_ents   = extract_entities(gt)
            missed = {}
            for tag, gt_vals in gt_ents.items():
                pred_vals = pred_ents.get(tag, [])
                missed_vals = [v for v in gt_vals
                               if not any(_covers(v, p) for p in pred_vals)]
                if missed_vals:
                    missed[tag] = missed_vals
            fn_rows.append({
                "idx": i,
                "completeness": round(score, 4),
                "missed_types": list(missed.keys()),
                "n_missed": sum(len(v) for v in missed.values()),
                "missed_detail": str(missed),
                "note_snippet": note[:120].replace('\n', ' '),
            })

fn_df = pd.DataFrame(fn_rows)
print(f"False Negatives (claimed=complete, actually missed PHI): {len(fn_df)}")
if not fn_df.empty:
    print(f"\nMost common missed entity types:")
    from collections import Counter
    type_counts = Counter(t for row in fn_rows for t in row['missed_types'])
    for t, c in type_counts.most_common():
        print(f"  {t:<12}: {c}")
fn_df.head(10)

## 10. Summary table for paper (Ablation Section)

Copy this into the paper's ablation table.

In [ ]:
# Build final summary
summary_rows = []
for label, r in stage_results.items():
    summary_rows.append({
        "Configuration":   label,
        "Micro-P":         f"{r.micro_precision:.4f}",
        "Micro-R":         f"{r.micro_recall:.4f}",
        "Micro-F1":        f"{r.micro_f1:.4f}",
        "F-beta(b=2)":     f"{r.f_beta:.4f}",
        "Macro-F1":        f"{r.macro_f1:.4f}",
        "Zero-leak":       f"{r.zero_leak_rate:.4f}",
        "Halluc.": f"{r.hallucination_rate:.4f}",
    })

# Add calibration row
summary_rows.append({
    "Configuration":   "[QC Calibration]",
    "Micro-P":  f"QC-Recall={calib.qc_recall:.4f}",
    "Micro-R":  f"QC-Prec={calib.qc_precision:.4f}",
    "Micro-F1": f"FN-rate={calib.fn_rate:.4f}",
    "F-beta(b=2)": f"FP-rate={calib.fp_rate:.4f}",
    "Macro-F1": "",
    "Zero-leak": f"tag-cov={calib.tag_coverage:.4f}",
    "Halluc.": "",
})

summary_df = pd.DataFrame(summary_rows)
print(summary_df.to_string(index=False))

# Save to CSV for easy copy-paste
os.makedirs(os.path.join(ROOT, 'artifacts', 'qc'), exist_ok=True)
out_path = os.path.join(ROOT, 'artifacts', 'qc', 'qc_ablation_summary.csv')
summary_df.to_csv(out_path, index=False)
print(f"\nSaved to {out_path}")

summary_df